In [1]:
#라이브러리 설치
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers chromadb transformers peft bitsandbytes accelerate lxml

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.3.2 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2025.10.0 which is incompatible.


In [4]:
import requests
from bs4 import BeautifulSoup
import torch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

def get_stock_news_retriever(ticker_code):
    """특정 주식 종목의 뉴스를 크롤링하여 검색기(Retriever)를 반환하는 함수"""
    print(f"\n🌎 [해외 증권] '{ticker_code}' 뉴스 데이터 수집 및 DB 구성 중...")

    keyword = ticker_code.split('.')[0] if '.' in ticker_code else ticker_code
    url = f"https://news.google.com/rss/search?q={keyword}+주가&hl=ko&gl=KR&ceid=KR:ko"

    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, "xml")
        items = soup.find_all("item")

        news_texts = []
        for item in items[:15]:
            title = item.title.text
            pub_date = item.pubDate.text
            news_texts.append(f"제목: {title} (발행일: {pub_date})")

        if not news_texts:
            print("⚠️ 검색된 뉴스가 없습니다.")
            return None

        full_text = "\n".join(news_texts)

        # 텍스트 분할 및 벡터 DB 구성
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
        splits = text_splitter.create_documents([full_text])

        embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
        db = Chroma.from_documents(splits, embeddings)

        # 중요: Retriever 객체를 반환해야 RAG 검색이 가능합니다.
        return db.as_retriever(search_kwargs={"k": 5}) # 관련도 높은 뉴스 5개 검색

    except Exception as e:
        print(f"⚠️ 크롤링 에러: {e}")
        return None

ImportError: dlopen(/opt/miniconda3/envs/llm/lib/python3.12/site-packages/torch/_C.cpython-312-darwin.so, 0x0002): Symbol not found: __ZN2at3mps14getMPSProfilerEv
  Referenced from: <40CDC379-70F7-3039-81FE-6159FD74FCDA> /opt/miniconda3/envs/llm/lib/libtorch_python.dylib
  Expected in:     <EF517B96-4CE4-357A-A9AB-03F243F0861C> /opt/miniconda3/envs/llm/lib/libtorch_cpu.dylib

In [ ]:
# 모델 및 LoRA 설정
model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4bit 양자화 설정 (메모리 최적화)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("1. Base Model 로드 중...")
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("2. LoRA(PEFT) 어댑터 설정 적용 중...")
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# 파인튜닝을 위한 LoRA 모델 래핑
lora_model = get_peft_model(base_model, peft_config)
print("LoRA 설정 완료. 학습 가능 파라미터:", lora_model.print_trainable_parameters())

# 주의: 여기서는 과제 요건을 맞추기 위해 셋업을 보여주지만,
# 실제로는 이 모델(lora_model)을 Trainer를 이용해 파인튜닝하는 과정이 필요합니다.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 17,432,576 || all params: 1,561,146,880 || trainable%: 1.1167
LoRA Trainable Parameters: None


In [ ]:
def predict_stock(ticker, retriever, model, tokenizer):
    print(f"\n[{ticker}] 종목 주가 예측을 위한 RAG 프로세스 시작...")

    # 1. RAG Retrieve: 질문과 관련된 문서를 벡터 DB에서 검색
    query = f"{ticker} 주가 상승 및 하락 요인 전망"
    docs = retriever.invoke(query)

    if not docs:
        return "관련 뉴스를 찾을 수 없어 예측이 불가능합니다."

    # 검색된 문서를 하나의 문자열로 결합 (Context 생성)
    context_text = "\n".join([f"- {doc.page_content}" for doc in docs])
    print(f"✅ RAG 검색 완료: {len(docs)}개의 문서를 프롬프트에 삽입합니다.")

    # 2. Prompt 구성
    prompt = f"""당신은 금융 시장을 분석하는 주식 전문가입니다.
다음 제공된 최신 뉴스 기사(Context)들을 바탕으로 {ticker} 종목의 주가를 오른다 혹은 내린다 둘중하나로 예측하세요.

[Context News]
{context_text}

[Question]
위 뉴스 기사에 근거하여 {ticker} 종목이 '오른다' 혹은 '내린다' 중 하나로 명확하게 결론을 내리고, 그 이유를 3줄 이내로 설명해주세요.

[Answer]
"""

    # 3. 모델 추론 (Generation)
    # 추론 시에는 모델을 평가 모드로 전환 (Dropout 등 비활성화)
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad(): # 추론 시 그래디언트 계산 비활성화 (메모리 절약)
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # 입력된 프롬프트 길이를 제외하고 새로 생성된 텍스트만 디코딩
    input_length = inputs['input_ids'].shape[1]
    result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return result

# 실행 예시 (종목: 알파벳A)
retriever = get_stock_news_retriever("GOOGL")
if retriever:
    prediction = predict_stock("GOOGL", retriever, lora_model, tokenizer)
    print("\n================== [예측 결과] ==================")
    print(prediction)

Searching news for GOOGL...

🌎 [해외 증권] 'GOOGL' 전용 뉴스 데이터 수집 중...
🔍 구글 뉴스(RSS)에서 'GOOGL' 관련 실제 뉴스를 가져옵니다...
👉 수집된 기사: 15건

    당신은 주식 전문가로 다음 뉴스기사들을 바탕으로 주가를 예측해주세요.

    [Context News]
    None

    [Question]
    뉴스기사에 근거하여 GOOGL 종목이 오를까요 내릴까요 명확하게 오른다 혹은 내린다 로 결정하여 대답해주세요.

    [Answer]
     내릴 것입니다. 
"""
GOOGL 주가는 최근 급격히 하락세를 보이고 있습니다. 이전 3일 동안의 시세를 분석해보면, 주가가 상승세를 보였던 날이 거의 없으며, 하락세가 확산되고 있다는 것을 알 수 있습니다. 특히 이번주에는 투자심리가 악화되면서 하락세가 더욱 가속화되었습니다. 따라서 GOOGL 주가는 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다. 따라서 GOOGL 주식을 추천하지 않습니다. 

따라서, GOOGL 주식은 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다.
```python
# 예측 결과 출력
print("GOOGL 주식은 내년第一季度 중반까지 계속해서 하락할 것으로 예상됩니다.")
```
